# 멈춘 동안 고치기 · updateState 와 사람 의견

강의 오후 ③ 가운데 (Academy 모듈 3 · L3 Editing State and Human Feedback)

| 칸 | 보여 주는 것 |
| --- | --- |
| ① State 고치기 | tutor 앞에서 멈추고 `updateState` 로 학생의 말을 하나 더 넣는다. 모델은 고친 State 를 본다 |
| ② 사람 의견 자리 | 아무 일도 안 하는 `human_feedback` 노드 앞에서 멈추고, 그 노드가 돈 것으로 쳐서(`asNode`) 의견을 넣는다 |

앞 노트북 `tutor-breakpoints.ipynb` 에서 멈추고, 보고, 이어 가는 법을 봤습니다. 이번에는 멈춰 있는 동안 State 를 고칩니다. 준비는 앞 노트북과 같습니다.

In [ ]:
import "npm:@langchain/core@1.2.14";
import { Annotation, StateGraph, START, END, MemorySaver } from "npm:@langchain/langgraph@1.4.18";

// 접속 정보: 실습과 같은 lab/.env 를 읽는다 (OpenRouter · qwen/qwen3-32b). 키는 화면에 찍지 않는다
const ENV: Record<string, string> = {};
try {
  for (const line of Deno.readTextFileSync("../lab/.env").split("\n")) {
    const m = line.match(/^\s*([A-Z_]+)\s*=\s*(.*)\s*$/);
    if (m) ENV[m[1]] = m[2];
  }
} catch { /* lab/.env 가 없으면 아래 기본값 */ }
const BASE_URL = ENV.LLM_BASE_URL ?? "https://openrouter.ai/api/v1";
const API_KEY  = ENV.LLM_API_KEY  ?? "";
const MODEL    = ENV.LLM_MODEL    ?? "qwen/qwen3-32b";
const OPTIONS  = { temperature: 0 };   // 시연이 매번 같게
console.log({ BASE_URL, MODEL, 키: API_KEY ? "있음" : "없음 — lab/.env 에 LLM_API_KEY 를 넣는다" });

// LangSmith (강사 시연용): demo/.env 에 키가 있으면 실행 기록(trace)이 LangSmith 에 남는다. 실습은 lab/.env 를 써서 남지 않는다
import { traceable } from "npm:langsmith@0.11.1/traceable";
try {
  for (const line of Deno.readTextFileSync(".env").split("\n")) {
    const m = line.match(/^\s*(LANGSMITH_[A-Z_]+)\s*=\s*(?:"([^"]*)"|([^#\s]*))/);   // "따옴표 값" 은 그대로, 아니면 # 주석 앞까지
    const value = m?.[2] ?? m?.[3];
    if (m && value) Deno.env.set(m[1], value);
  }
} catch { /* demo/.env 가 없으면 기록하지 않는다 */ }
if (!Deno.env.get("LANGSMITH_API_KEY")) Deno.env.set("LANGSMITH_TRACING", "false");   // 키가 없으면 끈다
// 모델을 부르는 함수를 감싸면 trace 에 "모델 호출" 한 칸으로 남는다. 기록을 안 켜면 그냥 원래 함수로 돈다
const asLLM = <F extends (...a: any[]) => Promise<any>>(fn: F, name: string) => traceable(fn, {
  name, run_type: "llm",
  processInputs: (i: any) => ({ messages: i.args ? i.args[0] : i.input }),
  processOutputs: (o: any) => ({ choices: [{ message: typeof o.outputs === "string" ? { role: "assistant", content: o.outputs } : o }] }),
});
console.log({ LangSmith: Deno.env.get("LANGSMITH_TRACING") === "true" && Deno.env.get("LANGSMITH_API_KEY") ? `기록함 · project ${Deno.env.get("LANGSMITH_PROJECT") ?? "default"}` : "기록 안 함 (demo/.env)" });

// 그래프 그림: 노드·엣지 이름을 mermaid.ink 로 보내 PNG 로 받아 셀 출력에 띄운다. 받지 못하면 mermaid 글자로 찍는다
async function showGraph(graph: any, options: { xray?: boolean } = {}) {
  const drawable = await graph.getGraphAsync(options);   // xray: true 면 subgraph 안의 노드까지 그린다
  try {
    const png = await drawable.drawMermaidPng();
    await Deno.jupyter.display(Deno.jupyter.image(new Uint8Array(await png.arrayBuffer())));
  } catch {
    console.log(drawable.drawMermaid());
  }
}

## 준비: 앞 노트북과 같은 튜터 Agent

모델 부르기, 사전 도구, 그래프(`builder`), 그리고 앞 노트북에서 stream 루프를 묶은 `run` 까지 그대로 준비합니다.

In [ ]:
// 오전에 만든 chat() 그대로 — 다시 보낼 칸(role · content · tool_calls)만 돌려준다
async function callModel(messages: any[], tools?: any[]) {
  const res = await fetch(`${BASE_URL}/chat/completions`, {
    method: "POST",
    headers: { "Content-Type": "application/json", Authorization: `Bearer ${API_KEY}` },
    body: JSON.stringify({ model: MODEL, messages, tools, ...OPTIONS }),
  });
  const { role, content, tool_calls } = (await res.json()).choices[0].message;
  return { role, content: content?.trim() ?? null, tool_calls };
}
const chat = asLLM(callModel, "chat");   // LangSmith 에 모델 호출로 남긴다

// 도구 하나: 단어 뜻과 쉬운 말 찾기 (작은 사전이라 시연이 매번 같다)
const DICT: Record<string, { 뜻: string; 쉬운말: string }> = {
  ubiquitous: { 뜻: "어디에나 있는", 쉬운말: "everywhere" },
  mitigate:   { 뜻: "완화하다, 줄이다", 쉬운말: "reduce" },
  meticulous: { 뜻: "꼼꼼한", 쉬운말: "careful" },
  detrimental:{ 뜻: "해로운", 쉬운말: "harmful" },
};
const tools = [{
  type: "function",
  function: {
    name: "lookup_word",
    description: "영어 단어 하나의 뜻과, 바꿔 쓸 쉬운 말을 사전에서 찾는다",
    parameters: { type: "object", properties: { word: { type: "string", description: "찾을 영어 단어 하나" } }, required: ["word"] },
  },
}];
function lookupWord(word: string) {
  const w = word.toLowerCase().trim();
  return DICT[w] ? { word: w, ...DICT[w] } : { word: w, error: "사전에 없는 단어" };
}

const SYSTEM = { role: "system", content: "너는 IELTS 쓰기를 돕는 영어 튜터다. 단어 뜻은 추측하지 말고 lookup_word 로 확인한다. 한국어로 짧게 답한다." };
const ask = (q: string) => ({ messages: [SYSTEM, { role: "user", content: q }] });
// 메시지를 보기 좋게 찍는다 (tutor-stream 노트북의 pretty 그대로)
const TITLE: Record<string, string> = { system: "System Message", user: "Human Message", assistant: "Ai Message", tool: "Tool Message" };
function pretty(m: any) {
  const title = ` ${TITLE[m.role]} `;
  const left = Math.floor((60 - title.length) / 2);
  console.log("=".repeat(left) + title + "=".repeat(60 - title.length - left));
  if (m.tool_calls?.length) {
    const calls = m.tool_calls.map((c: any) => `${c.function.name}(${c.function.arguments})`);
    console.log("Tool Calls:", calls.join(", "));
  } else console.log(m.content);
}

In [ ]:
// 오후 ① 의 튜터 Agent 그대로: tutor ⇢ tools → tutor
const State = Annotation.Root({
  messages: Annotation<any[]>({ reducer: (prev, next) => [...prev, ...next], default: () => [] }),
});
async function tutor(state: typeof State.State) {
  return { messages: [await chat(state.messages, tools)] };
}
function toolsNode(state: typeof State.State) {
  const calls = state.messages.at(-1).tool_calls;
  console.log(`▶ tools   ${calls.map((c: any) => c.function.arguments).join(" · ")}`);
  return { messages: calls.map((c: any) => ({
    role: "tool", tool_call_id: c.id, content: JSON.stringify(lookupWord(JSON.parse(c.function.arguments).word)),
  })) };
}
function route(state: typeof State.State) {
  return state.messages.at(-1).tool_calls?.length ? "tools" : END;
}
const builder = new StateGraph(State)
  .addNode("tutor", tutor)
  .addNode("tools", toolsNode)
  .addEdge(START, "tutor")
  .addConditionalEdges("tutor", route, ["tools", END])
  .addEdge("tools", "tutor");

// 앞 노트북의 run 그대로: stream 으로 돌리며 State 를 받을 때마다 찍고, 멈추면 표시한다
async function run(graph: any, input: any, thread: any) {
  for await (const state of await graph.stream(input, { ...thread, streamMode: "values" })) {
    if (state.__interrupt__) console.log("⏸ __interrupt__ · 다음 노드:", (await graph.getState(thread)).next);
    else {
      for (const m of state.messages) if (m.role !== "system") pretty(m);   // system 메시지는 매번 같으니 뺀다
      console.log("-".repeat(60));   // State 하나 끝
    }
  }
}

## ① 멈춘 동안 State 를 고친다 — updateState

멈춰 있는 동안에는 State 를 볼 수만 있는 게 아니라 **고칠** 수도 있습니다. 이번에는 tutor 앞에서 멈춥니다. 모델을 부르기 전입니다.

학생이 ubiquitous 를 물었다가 "아니, meticulous 를 찾아 줘" 라고 바꿨다고 해 보겠습니다. `updateState(thread, 값)` 으로 State 에 값을 넣습니다. 노드가 돌려준 값과 똑같이 그 필드의 reducer 를 거칩니다. messages 는 이어 붙이는 reducer 니까 뒤에 붙습니다.

In [ ]:
// ① 멈춘 동안 State 를 고친다: tutor 앞에서 멈추고, 학생의 말을 하나 더 넣는다
const editAgent = builder.compile({ checkpointer: new MemorySaver(), interruptBefore: ["tutor"] });
await showGraph(editAgent);
const thread3 = { configurable: { thread_id: "edit-state-1" } };
await run(editAgent, ask("ubiquitous 뜻을 사전에서 찾아 줘"), thread3);   // 모델을 부르기 전, tutor 앞에서 멈춘다
await editAgent.updateState(thread3, { messages: [{ role: "user", content: "아니, ubiquitous 말고 meticulous 를 찾아 줘" }] });
for (const m of (await editAgent.getState(thread3)).values.messages) console.log(`  ${m.role}: ${m.content}`);

**돌려 보면** 아직 모델을 부르지 않았는데 user 메시지가 두 개입니다. 이제 이어 갑니다. tutor 앞에서 멈추게 해 뒀으니, tools 를 돌고 나서 다시 tutor 앞에서 멈춥니다. 그래서 null 로 두 번 이어 갑니다.

In [ ]:
// 이어 가면 모델은 고친 State 를 본다. tutor → tools 를 돌고, 다시 tutor 앞에서 멈춘다
await run(editAgent, null, thread3);

tutor 앞에서 또 멈췄습니다. null 로 한 번 더 이어 갑니다.

In [ ]:
// 한 번 더 이어 가면 tutor 가 답한다
await run(editAgent, null, thread3);

**돌려 보면** 모델은 고친 State 를 보고 meticulous 를 찾습니다. 답도 meticulous 입니다.

## ② 사람 의견을 넣는 자리 — human_feedback 노드

①에서는 tutor 앞에서 멈췄습니다. 사람 의견을 넣는 자리를 그래프에 아예 **노드로** 만들어 둘 수도 있습니다. 아무 일도 하지 않는 `human_feedback` 노드를 tutor 앞에 두고, 그 앞에서 멈춥니다.

의견을 넣을 때는 `updateState` 의 세 번째 인자로 `"human_feedback"` 을 줍니다(asNode). **그 노드가 이 값을 돌려준 것으로 친다**는 뜻입니다. 그래서 human_feedback 은 돈 것이 되고, 이어 가면 다음 노드인 tutor 부터 돕니다.

In [ ]:
// ② 사람 의견을 넣는 자리: 아무 일도 하지 않는 human_feedback 노드를 두고, 그 앞에서 멈춘다
const feedbackAgent = new StateGraph(State)
  .addNode("human_feedback", () => ({}))      // 하는 일 없음. 사람이 이 노드인 셈 치고 State 를 고친다
  .addNode("tutor", tutor)
  .addNode("tools", toolsNode)
  .addEdge(START, "human_feedback")
  .addEdge("human_feedback", "tutor")
  .addConditionalEdges("tutor", route, ["tools", END])
  .addEdge("tools", "human_feedback")
  .compile({ checkpointer: new MemorySaver(), interruptBefore: ["human_feedback"] });
await showGraph(feedbackAgent);
const thread4 = { configurable: { thread_id: "edit-state-2" } };
await run(feedbackAgent, ask("ubiquitous 뜻을 사전에서 찾아 줘"), thread4);   // 시작하자마자 human_feedback 앞에서 멈춘다
await feedbackAgent.updateState(thread4, { messages: [{ role: "user", content: "아니, meticulous 를 찾아 줘" }] }, "human_feedback");
console.log("다음 노드:", (await feedbackAgent.getState(thread4)).next);   // human_feedback 이 돈 것으로 쳤다 → 다음은 tutor

이제 이어 갑니다. human_feedback 이 돈 것으로 쳤으니 tutor 부터 돕니다.

In [ ]:
// 이어 가기: tutor → tools → human_feedback 앞에서 멈춤
await run(feedbackAgent, null, thread4);

**돌려 보면** 처음에는 다음 노드가 `human_feedback`, 의견을 넣은 뒤에는 `tutor` 입니다. 이어 가면 tutor 가 meticulous 를 찾고, tools 를 돈 뒤에 다시 human_feedback 앞에서 멈춥니다.

더 고칠 게 없으면 State 를 건드리지 않고 그냥 이어 갑니다.

In [ ]:
// 더 고칠 게 없으면 State 를 건드리지 않고 그냥 이어 간다
await run(feedbackAgent, null, thread4);

**돌려 보면** tutor 가 답하고 끝납니다.

asNode 를 안 주면 어떻게 될까요? 그때는 LangGraph 가 "마지막으로 돈 노드가 쓴 것" 으로 치고, 그 노드 다음에 갈 자리를 다시 정합니다. 여기서 마지막으로 돈 노드는 tools 이고 tools 다음은 human_feedback 이니, 다시 human_feedback 앞에 섭니다. 어느 노드가 쓴 것으로 칠지에 따라 이어 갈 자리가 달라집니다.